# 实验 02 · 特征缩放与数据泄漏

目标：只在训练集拟合预处理参数，并正确处理常量特征。

本实验为新增练习，仅依赖 NumPy；题干、测试均不包含参考实现。
从上到下执行，完成每个 TODO 后运行对应自测。

In [1]:
import numpy as np
rng = np.random.default_rng(42)

## 1 · 拟合标准化参数

实现 fit_standardizer(X_train)，返回 (mu,scale)，形状都为 (n,)。使用总体标准差（ddof=0）；常量列的 scale 设为 1。禁止原地修改数据。

In [4]:
def fit_standardizer(X_train):

    X_train = np.asarray(X_train)
    mu = np.mean(X_train, axis=0)
    scale = np.std(X_train, axis=0, ddof=0)
    scale = np.where(scale == 0, 1.0, scale)
    return mu, scale

    raise NotImplementedError("TODO: 拟合均值与尺度")

In [5]:
X = np.array([[1., 10., 4.], [3., 30., 4.]])
before = X.copy()
mu, scale = fit_standardizer(X)
np.testing.assert_allclose(mu, [2., 20., 4.])
np.testing.assert_allclose(scale, [1., 10., 1.])
np.testing.assert_array_equal(X, before)
assert np.asarray(mu).shape == (3,) and np.asarray(scale).shape == (3,)
print("通过：总体标准差、常量列、输入未修改")

通过：总体标准差、常量列、输入未修改


## 2 · 转换与逆转换

实现 transform(X,mu,scale) 与 inverse_transform(Z,mu,scale)。验证集必须复用训练集参数，不能重新拟合。两者均返回新数组。

In [6]:
def transform(X, mu, scale):

    X = np.asarray(X)
    return (X - mu) / scale

    raise NotImplementedError("TODO: 标准化转换")

def inverse_transform(Z, mu, scale):

    Z = np.asarray(Z)
    return Z * scale + mu

    raise NotImplementedError("TODO: 逆转换")

In [7]:
train = np.array([[1., 10., 4.], [3., 30., 4.]])
val = np.array([[100., 1000., 4.]])
mu, scale = fit_standardizer(train)
Z = transform(train, mu, scale)
np.testing.assert_allclose(Z, [[-1., -1., 0.], [1., 1., 0.]])
np.testing.assert_allclose(transform(val, mu, scale), [[98., 98., 0.]])
np.testing.assert_allclose(inverse_transform(Z, mu, scale), train)
assert not np.shares_memory(Z, train)
print("通过：训练/验证参数复用、逆转换")

通过：训练/验证参数复用、逆转换


## 实验记录

写出输入/输出形状，解释一个边界情况，再记录一次修改参数前后的观察。

`fit_standardizer(X_train)`：输入形状 `(m, n)`，输出 `mu`、`scale` 形状均为 `(n,)`。
`transform(X, mu, scale)`：输入形状 `(m, n)`，输出形状 `(m, n)`。
`inverse_transform(Z, mu, scale)`：输入形状 `(m, n)`，输出形状 `(m, n)`。

常量列的标准差为 0，若直接作为分母会产生除零；因此将常量列的 `scale` 设为 1。此时该列标准化后为 0，逆转换可恢复原常量值。

验证集必须复用训练集拟合得到的 `mu` 和 `scale`，不能重新拟合，否则会造成数据泄漏。例如验证值 `100` 使用训练集参数标准化后为 `98`，说明它相对于训练分布被正确转换。